In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.metrics import mean_squared_error,r2_score

In [2]:
df=pd.read_csv("1.04. Real-life example.csv")
df.head(5)

,Brand,Price,Body,Mileage,EngineV,Engine Type,Registration,Year,Model
0,BMW,4200.0,sedan,277,2.0,Petrol,yes,1991,320
1,Mercedes-Benz,7900.0,van,427,2.9,Diesel,yes,1999,Sprinter 212
2,Mercedes-Benz,13300.0,sedan,358,5.0,Gas,yes,2003,S 500
3,Audi,23000.0,crossover,240,4.2,Petrol,yes,2007,Q7
4,Toyota,18300.0,crossover,120,2.0,Petrol,yes,2011,Rav 4


In [3]:
df=df.dropna(subset=['Price','EngineV'])
df=df[df['EngineV']<=10]

In [4]:
df['Log_price']=np.log(df['Price'])

In [5]:
cat_cols=df.select_dtypes(include='object').columns.tolist()
encoders={}
for col in cat_cols:
    le=LabelEncoder()
    df[col]=le.fit_transform(df[col])
    encoders[col]=le
joblib.dump(encoders,"encoders.pkl")    
joblib.dump(cat_cols,"cat_cols.pkl")    


['cat_cols.pkl']

In [6]:
X=df.drop(['Log_price','Price'],axis=1)
Y=df['Log_price']

In [ ]:
feature_columns=X.columns.tolist()

joblib.dump(feature_columns,"feature_columns.pkl")

['feature_columns.pkl']

In [14]:
feature_columns

['Brand',
 'Body',
 'Mileage',
 'EngineV',
 'Engine Type',
 'Registration',
 'Year',
 'Model']

In [17]:
print(cat_cols)
print(encoders)

['Brand', 'Body', 'Engine Type', 'Registration', 'Model']
{'Brand': LabelEncoder(), 'Body': LabelEncoder(), 'Engine Type': LabelEncoder(), 'Registration': LabelEncoder(), 'Model': LabelEncoder()}


In [8]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [9]:
models={
    "Linear Regression":LinearRegression(),
    "Ridge":Ridge(alpha=1.0,random_state=42),
    "Lasso":Lasso(alpha=1.0,random_state=42),
}

In [10]:
results=[]
fitted={}
for name,model in models.items():
    model.fit(X_train,Y_train)
    preds=model.predict(X_test)
    rmse=np.sqrt(mean_squared_error(Y_test,preds))
    r2=r2_score(Y_test,preds)
    results.append({"Model":name,"RMSE":rmse,"R2":r2})
    fitted[name]=model
results_df=pd.DataFrame(results).sort_values("RMSE")
print(results_df) 
best_name=results_df.iloc[0]["Model"]
best_model=fitted[best_name]
print(f"\nBest model:{best_name}")   

               Model      RMSE        R2
0  Linear Regression  0.387991  0.829826
1              Ridge  0.387996  0.829822
2              Lasso  0.654229  0.516153

Best model:Linear Regression


In [11]:
joblib.dump(best_model,"car_price_model.pkl")

['car_price_model.pkl']

In [12]:
numeric_cols=[c for c in feature_columns if c not in cat_cols]
ranges={c:(float(df[c].min()),float(df[c].max())) for c in numeric_cols}
joblib.dump(ranges,"numeric_ranges.pkl")

['numeric_ranges.pkl']

In [13]:
print("\nSaved: car_price_model.pkl,encoders.pkl,cat_cols.pkl,feature_columns.pkl,numeric_ranges.pkl")
print("Copy all 5 .pkl files into the same folder as copy.py before deploying. ")


Saved: car_price_model.pkl,encoders.pkl,cat_cols.pkl,feature_columns.pkl,numeric_ranges.pkl
Copy all 5 .pkl files into the same folder as copy.py before deploying. 
